# NB10 Pilot — Custom Loss Functions (BCE vs Sigmoid-MSE)

**Purpose:** Diagnostic pilot, NOT a thesis-numbered notebook. Follow-up to `NB10_pilot_logitdistloss.ipynb`, which confirmed `elementwise_loss="LogitDistLoss()"` is a robust *regression* loss (not cross-entropy) and cannot produce calibrated probabilities regardless of search budget (confirmed at both 400 and 6000 iterations).

Since PySR/SymbolicRegression.jl has no built-in cross-entropy loss, this notebook tests two **custom Julia loss functions**, passed directly via `elementwise_loss`, per the supervisor's specification (2026-06-25):

- **Pilot A — Binary cross-entropy (BCE):** `-(t*log(p) + (1-t)*log(1-p))` with `p = sigmoid(pred)` computed inside the loss. Theoretically optimal for probability estimation; may be less stable for evolutionary search due to large gradients when confidently wrong.
- **Pilot B — Sigmoid-MSE:** `(sigmoid(pred) - t)^2`. Keeps the bounded, quadratic gradient landscape PySR was originally tuned for, while still forcing the search into logit space via the sigmoid wrapper.

**Four pass/fail diagnostic checks per pilot** (supervisor's criteria):
1. Complexity-1 constant should be **~-1.6** (logit of the 16.9% base rate), not ~+0.16.
2. Raw (logit) output should be **mostly negative** (since 83% of patients survive).
3. Sigmoid (probability) output range should dip **well below 0.50**.
4. Validation ECE (after sigmoid) should be **below 0.15** at this reduced 400-iteration budget.

**No clipping anywhere** — sigmoid guarantees (0,1) bounds for any finite input by construction.

**Settings (both pilots):** niterations=400, populations=30, maxsize=24, parsimony=0.001, same operators as canonical, random_state=42 — identical except for `elementwise_loss`.

**Output:** `results/gp_runs_pilot_customloss/pilot_a_bce_model.pkl`, `results/gp_runs_pilot_customloss/pilot_b_sigmoidmse_model.pkl` — diagnostic only, not thesis artefacts.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json, pickle, time, sys, warnings
warnings.filterwarnings("ignore")

_nb_dir   = Path().resolve()
PROJECT   = _nb_dir.parent if _nb_dir.name == "notebooks" else _nb_dir
DATA_PROC = PROJECT / "data" / "processed"
PILOT_DIR = PROJECT / "results" / "gp_runs_pilot_customloss"
PILOT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from src.metrics import compute_ece, compute_metrics

feat = pd.read_parquet(DATA_PROC / "features_curated.parquet")
with open(DATA_PROC / "feature_config.json") as f:
    cfg = json.load(f)
GP_TERMINALS = cfg["GP_TERMINALS"]

split_df = pd.read_csv(DATA_PROC / "split_random.csv")
feat_split = feat.merge(split_df[["patientunitstayid", "split"]], on="patientunitstayid")

train_df = feat_split[feat_split["split"] == "train"].reset_index(drop=True)
val_df   = feat_split[feat_split["split"] == "val"].reset_index(drop=True)
test_df  = feat_split[feat_split["split"] == "test"].reset_index(drop=True)

X_train_gp = train_df[GP_TERMINALS].copy()
y_train    = train_df["hospital_mortality"].to_numpy(dtype=np.float64)
X_val_gp   = val_df[GP_TERMINALS].copy()
y_val      = val_df["hospital_mortality"].to_numpy(dtype=np.float64)
X_test_gp  = test_df[GP_TERMINALS].copy()
y_test     = test_df["hospital_mortality"].to_numpy(dtype=np.float64)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

TARGET_LOGIT = float(np.log(y_train.mean() / (1 - y_train.mean())))
print(f"Train: {len(train_df):,} ({y_train.mean()*100:.2f}%)  "
      f"Val: {len(val_df):,} ({y_val.mean()*100:.2f}%)  "
      f"Test: {len(test_df):,} ({y_test.mean()*100:.2f}%)")
print(f"GP_TERMINALS: {len(GP_TERMINALS)} features")
print(f"Target complexity-1 constant (logit of train base rate): {TARGET_LOGIT:.4f}")
print(f"Pilot output dir: {PILOT_DIR}")

Train: 7,814 (16.88%)  Val: 1,117 (16.92%)  Test: 2,233 (16.88%)
GP_TERMINALS: 26 features
Target complexity-1 constant (logit of train base rate): -1.5942
Pilot output dir: C:\ML PROJECT\sepsis-gp\results\gp_runs_pilot_customloss


In [2]:
print("Importing PySR -- Julia compilation may take 5-10 min on first run ...")
t_import = time.time()
from pysr import PySRRegressor
print(f"PySR import complete in {time.time()-t_import:.0f}s.")

Importing PySR -- Julia compilation may take 5-10 min on first run ...
Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
PySR import complete in 11s.


In [4]:
# ============================================================================
# Shared diagnostic + Pareto-evaluation helper for both pilots -- evaluates
# the Pareto front on validation (sigmoid-transformed), selects best by
# fitness (AUROC - 0.5*ECE), checks test set, and runs the four supervisor
# diagnostic criteria.
# ============================================================================
def evaluate_pilot(gp_model, label):
    pareto_rows = []
    for idx, eq_row in gp_model.equations_.iterrows():
        try:
            raw_val = gp_model.predict(X_val_gp, index=idx)
            raw_val = np.where(np.isfinite(raw_val), raw_val, 0.0)
            prob_val = sigmoid(raw_val)
            m = compute_metrics(y_val, prob_val, label=f"c{int(eq_row['complexity'])}",
                                 fitness_alpha=1.0, fitness_beta=0.5)
            m["complexity"] = int(eq_row["complexity"])
            m["equation"]   = str(eq_row["equation"])
            m["eq_idx"]     = idx
            m["raw_min"]    = float(raw_val.min())
            m["raw_max"]    = float(raw_val.max())
            m["prob_min"]   = float(prob_val.min())
            m["prob_max"]   = float(prob_val.max())
            m["pct_raw_negative"] = float((raw_val < 0).mean() * 100)
            pareto_rows.append(m)
        except Exception as e:
            print(f"  c={eq_row['complexity']} skipped: {e}")

    pareto_df = pd.DataFrame(pareto_rows)
    print(f"\n=== {label}: Pareto front (validation set) ===")
    print(pareto_df[["complexity", "auroc", "ece_10bin", "brier", "fitness",
                     "raw_min", "raw_max", "prob_min", "prob_max"]].to_string(index=False))

    best_idx_row = pareto_df["fitness"].idxmax()
    best = pareto_df.loc[best_idx_row]
    eq_idx = int(best["eq_idx"])

    c1_row = pareto_df[pareto_df["complexity"] == 1]
    c1_raw = float(c1_row["raw_min"].iloc[0]) if len(c1_row) else None

    raw_test  = gp_model.predict(X_test_gp, index=eq_idx)
    raw_test  = np.where(np.isfinite(raw_test), raw_test, 0.0)
    prob_test = sigmoid(raw_test)
    test_m = compute_metrics(y_test, prob_test, label=f"{label}_test")

    print(f"\nSelected by val fitness: complexity={int(best['complexity'])}")
    print(f"  Equation: {best['equation']}")
    print(f"  Raw (logit) range on val: [{best['raw_min']:.4f}, {best['raw_max']:.4f}]")
    print(f"  Sigmoid (prob) range on val: [{best['prob_min']:.6f}, {best['prob_max']:.6f}]")
    print(f"\nTest set (n={len(y_test):,}):")
    print(f"  AUROC: {test_m['auroc']:.4f}  ECE: {test_m['ece_10bin']:.4f}  Brier: {test_m['brier']:.4f}")
    print(f"  Raw (logit) range on test: [{raw_test.min():.4f}, {raw_test.max():.4f}]")
    print(f"  Sigmoid (prob) range on test: [{prob_test.min():.6f}, {prob_test.max():.6f}]")

    print(f"\n--- Diagnostic checks ({label}) ---")
    check1 = (c1_raw is not None) and (abs(c1_raw - TARGET_LOGIT) < 0.5)
    print(f"  1. Complexity-1 constant ~ {TARGET_LOGIT:.2f}?  Got: {c1_raw}  -> {'PASS' if check1 else 'FAIL'}")
    check2 = best["pct_raw_negative"] > 50 or pareto_df["pct_raw_negative"].mean() > 50
    print(f"  2. Raw output mostly negative?  Selected eq: {best['pct_raw_negative']:.1f}% negative on val  "
          f"-> {'PASS' if check2 else 'FAIL'}")
    check3 = prob_test.min() < 0.40
    print(f"  3. Prob range dips well below 0.50?  Test min={prob_test.min():.4f}  "
          f"-> {'PASS' if check3 else 'FAIL'}")
    check4 = test_m["ece_10bin"] < 0.15
    print(f"  4. Val/test ECE < 0.15?  Test ECE={test_m['ece_10bin']:.4f}  -> {'PASS' if check4 else 'FAIL'}")

    overall_pass = check1 and check2 and check3 and check4
    print(f"\n  OVERALL: {'PASS' if overall_pass else 'FAIL'}")

    return dict(label=label, pareto_df=pareto_df, best=best, test_m=test_m,
                checks=dict(c1=check1, c2=check2, c3=check3, c4=check4, overall=overall_pass),
                raw_test_range=(float(raw_test.min()), float(raw_test.max())),
                prob_test_range=(float(prob_test.min()), float(prob_test.max())))

In [5]:
# ============================================================================
# PILOT A -- Custom binary cross-entropy, sigmoid applied inside the loss.
# ============================================================================
BCE_LOSS = ("((pred, tgt) -> let p = 1.0/(1.0+exp(-pred)); "
            "p = clamp(p, 1e-10, 1.0-1e-10); "
            "-(tgt*log(p) + (1.0-tgt)*log(1.0-p)) end)")

gp_a = PySRRegressor(
    niterations      = 400,
    populations      = 30,
    maxsize          = 24,
    parsimony        = 0.001,
    binary_operators = ["+", "-", "*", "/", "max", "min"],
    unary_operators  = ["log", "sqrt", "exp", "abs"],
    elementwise_loss  = BCE_LOSS,
    model_selection  = "best",
    random_state     = 42,
    verbosity        = 0,
    progress         = False,
    output_directory = str(PILOT_DIR / "pilot_a"),
)

print("Pilot A: starting BCE fit (niterations=400, populations=30) ...")
t0 = time.time()
gp_a.fit(X_train_gp, y_train)
print(f"Pilot A fit complete in {(time.time()-t0)/60:.1f} min.")

with open(PILOT_DIR / "pilot_a_bce_model.pkl", "wb") as f:
    pickle.dump(gp_a, f)
print(f"Saved: {PILOT_DIR / 'pilot_a_bce_model.pkl'}")

Pilot A: starting BCE fit (niterations=400, populations=30) ...
Pilot A fit complete in 8.5 min.
Saved: C:\ML PROJECT\sepsis-gp\results\gp_runs_pilot_customloss\pilot_a_bce_model.pkl


In [6]:
result_a = evaluate_pilot(gp_a, "Pilot A (BCE)")


=== Pilot A (BCE): Pareto front (validation set) ===
 complexity  auroc  ece_10bin  brier  fitness   raw_min   raw_max  prob_min  prob_max
          1 0.5000     0.0004 0.1406   0.4998 -1.594125 -1.594125  0.168804  0.168804
          3 0.5650     0.0159 0.1390   0.5571 -1.837239 -0.837239  0.137378  0.302116
          4 0.6749     0.0226 0.1276   0.6636 -3.688879  1.916923  0.024390  0.871795
          5 0.6737     0.0137 0.1269   0.6669 -3.386650  3.292887  0.032715  0.964184
          6 0.6754     0.0174 0.1280   0.6667 -2.960763  2.305761  0.049230  0.909353
          7 0.6910     0.0090 0.1280   0.6865 -3.450118  1.657627  0.030765  0.839919
          8 0.6967     0.0204 0.1283   0.6865 -4.092293  1.302156  0.016427  0.786198
          9 0.6964     0.0284 0.1289   0.6822 -3.355440  1.742092  0.033717  0.850953
         10 0.7137     0.0226 0.1269   0.7024 -3.196649  2.362326  0.039292  0.913909
         12 0.7193     0.0215 0.1266   0.7086 -3.193976  6.916549  0.039393  0.999010


In [7]:
# ============================================================================
# PILOT B -- Sigmoid-MSE: sigmoid(pred) compared to target via squared error.
# ============================================================================
SIGMOID_MSE_LOSS = "((pred, tgt) -> (1.0/(1.0+exp(-pred)) - tgt)^2)"

gp_b = PySRRegressor(
    niterations      = 400,
    populations      = 30,
    maxsize          = 24,
    parsimony        = 0.001,
    binary_operators = ["+", "-", "*", "/", "max", "min"],
    unary_operators  = ["log", "sqrt", "exp", "abs"],
    elementwise_loss  = SIGMOID_MSE_LOSS,
    model_selection  = "best",
    random_state     = 42,
    verbosity        = 0,
    progress         = False,
    output_directory = str(PILOT_DIR / "pilot_b"),
)

print("Pilot B: starting Sigmoid-MSE fit (niterations=400, populations=30) ...")
t0 = time.time()
gp_b.fit(X_train_gp, y_train)
print(f"Pilot B fit complete in {(time.time()-t0)/60:.1f} min.")

with open(PILOT_DIR / "pilot_b_sigmoidmse_model.pkl", "wb") as f:
    pickle.dump(gp_b, f)
print(f"Saved: {PILOT_DIR / 'pilot_b_sigmoidmse_model.pkl'}")

Pilot B: starting Sigmoid-MSE fit (niterations=400, populations=30) ...
Pilot B fit complete in 5.1 min.
Saved: C:\ML PROJECT\sepsis-gp\results\gp_runs_pilot_customloss\pilot_b_sigmoidmse_model.pkl


In [8]:
result_b = evaluate_pilot(gp_b, "Pilot B (Sigmoid-MSE)")


=== Pilot B (Sigmoid-MSE): Pareto front (validation set) ===
 complexity  auroc  ece_10bin  brier  fitness    raw_min   raw_max  prob_min  prob_max
          1 0.5000     0.0004 0.1406   0.4998  -1.594153 -1.594153  0.168800  0.168800
          3 0.6480     0.0331 0.1312   0.6314 -11.958422 -0.149480  0.000006  0.462699
          4 0.6480     0.0185 0.1278   0.6387  -2.643698  1.707559  0.066378  0.846519
          6 0.6707     0.0164 0.1284   0.6625  -2.944298  2.406959  0.050007  0.917356
          7 0.6943     0.0186 0.1266   0.6850  -3.306516  3.225916  0.035348  0.961798
          8 0.7046     0.0187 0.1258   0.6953  -2.959356  1.632964  0.049296  0.836575
          9 0.6999     0.0178 0.1270   0.6910  -3.750283  3.483607  0.022971  0.970218
         10 0.7144     0.0200 0.1262   0.7044  -3.247223  2.345096  0.037427  0.912544
         12 0.7229     0.0171 0.1252   0.7143  -3.149607  2.142712  0.041107  0.894986
         14 0.7256     0.0203 0.1215   0.7155  -3.298853  1.683750  

In [ ]:
# ============================================================================
# Side-by-side comparison
# - both pass -> pick better val ECE/AUROC
# - one passes -> use that one
# - neither passes -> report and diagnose further
# ============================================================================
print("=" * 70)
print("SIDE-BY-SIDE COMPARISON")
print("=" * 70)
for r in [result_a, result_b]:
    print(f"\n{r['label']}:")
    print(f"  Test AUROC={r['test_m']['auroc']:.4f}  ECE={r['test_m']['ece_10bin']:.4f}  "
          f"Brier={r['test_m']['brier']:.4f}")
    print(f"  Raw range: {r['raw_test_range']}  Prob range: {r['prob_test_range']}")
    print(f"  Checks: c1={r['checks']['c1']} c2={r['checks']['c2']} "
          f"c3={r['checks']['c3']} c4={r['checks']['c4']}  OVERALL={r['checks']['overall']}")

a_pass, b_pass = result_a["checks"]["overall"], result_b["checks"]["overall"]
print("\n" + "=" * 70)
if a_pass and b_pass:
    winner = "Pilot A (BCE)" if result_a["test_m"]["ece_10bin"] <= result_b["test_m"]["ece_10bin"] else "Pilot B (Sigmoid-MSE)"
    print(f"DECISION: Both pass. Better val/test ECE: {winner}. Recommend full 30-run marathon with this loss.")
elif a_pass:
    print("DECISION: Pilot A (BCE) passes, Pilot B fails. Recommend full marathon with BCE loss.")
elif b_pass:
    print("DECISION: Pilot B (Sigmoid-MSE) passes, Pilot A fails. Recommend full marathon with Sigmoid-MSE loss.")
else:
    print("DECISION: Neither pilot passes all four checks. Do NOT proceed to full marathon -- ")
    print("report results to supervisor for further diagnosis before continuing.")
print("=" * 70)

SIDE-BY-SIDE COMPARISON

Pilot A (BCE):
  Test AUROC=0.7449  ECE=0.0139  Brier=0.1192
  Raw range: (-3.4275726658311134, 4.185166690606045)  Prob range: (0.03144477511992934, 0.9850084969558166)
  Checks: c1=True c2=True c3=True c4=True  OVERALL=True

Pilot B (Sigmoid-MSE):
  Test AUROC=0.7141  ECE=0.0111  Brier=0.1232
  Raw range: (-4.0172185825467706, 4.3137704989922)  Prob range: (0.01768459393963829, 0.9867937454433215)
  Checks: c1=True c2=True c3=True c4=True  OVERALL=True

DECISION: Both pass. Better val/test ECE: Pilot B (Sigmoid-MSE). Recommend full 30-run marathon with this loss.


## Findings

*[To be completed once Cells 1-8 have been run. Record: (1) wall-clock time for each pilot, (2) full Pareto tables and diagnostic check results for both pilots, (3) the selected formula and structure for whichever pilot(s) pass, (4) the final decision per the supervisor's rule, (5) recommended next step -- either proceed to the full 30-run marathon with the winning loss, or report back to supervisor if neither pilot passes.]*